# 🎯 Stage 4: Final Comparison & Analysis

**Objective**: So sánh và phân tích tất cả các approaches:
- **Stage 1**: Weak Supervision (Reddit Gaming)
- **Stage 2**: Supervised Learning (Balanced Dataset)  
- **Stage 3a**: Supervised + Focal Loss (Imbalanced)
- **Stage 3b**: Supervised + Class Weighting (Imbalanced)

This notebook provides comprehensive comparison, visualizations, và recommendations.

## 📦 Environment Setup

In [ ]:
# Install required libraries
!pip install -q pandas numpy matplotlib seaborn plotly scikit-learn

import warnings
warnings.filterwarnings('ignore')

## 📚 Import Libraries

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from google.colab import files

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

print("✅ Libraries imported successfully!")

## 📂 Load Results from All Stages

Upload all result JSON files:
- `stage1_results.json` - Weak Supervision
- `stage2_results.json` - Balanced Supervised
- `stage3_focal_results.json` - Focal Loss
- `stage3_weighted_results.json` - Class Weighting

In [ ]:
print("📤 Please upload all result JSON files...")
uploaded = files.upload()

# Load all results
results = {}
stage_names = {
    'stage1': 'Weak Supervision\n(Reddit Gaming)',
    'stage2': 'Supervised\n(Balanced)',
    'stage3_focal': 'Focal Loss\n(Imbalanced)',
    'stage3_weighted': 'Class Weighting\n(Imbalanced)'
}

for filename in uploaded.keys():
    with open(filename, 'r') as f:
        data = json.load(f)
        
    # Determine stage from filename or content
    if 'stage1' in filename:
        results['stage1'] = data
    elif 'stage2' in filename:
        results['stage2'] = data
    elif 'focal' in filename or 'stage3' in filename:
        if 'weighted' in filename or 'weight' in filename:
            results['stage3_weighted'] = data
        else:
            results['stage3_focal'] = data

print(f"\n✅ Loaded {len(results)} stage results:")
for stage in results.keys():
    print(f"   - {stage}: {stage_names.get(stage, stage)}")

## 📊 Create Comparison DataFrame

In [ ]:
# Extract key metrics from all stages
comparison_data = []

for stage_key, result in results.items():
    comparison_data.append({
        'Stage': stage_names.get(stage_key, stage_key),
        'Method': result.get('method', 'Unknown'),
        'Accuracy (%)': result['metrics']['accuracy'] * 100,
        'F1-Score (%)': result['metrics']['f1_weighted'] * 100,
        'Training Time (min)': result.get('training_time_seconds', 0) / 60,
        'Dataset Size': result.get('dataset_size', result.get('train_size', 0)),
        'Train Size': result.get('train_size', 0),
        'Model': result.get('model', 'Unknown'),
        'Manual Labels': 'No' if 'weak' in stage_key.lower() else 'Yes'
    })

df_comparison = pd.DataFrame(comparison_data)

# Sort by accuracy descending
df_comparison = df_comparison.sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)

print("📊 Comparison Table:")
print("=" * 120)
display(df_comparison)
print("=" * 120)

## 📈 Visualization 1: Accuracy & F1-Score Comparison

In [ ]:
# Create comparison bar chart
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Accuracy Comparison', 'F1-Score Comparison'),
    horizontal_spacing=0.15
)

stages = df_comparison['Stage'].tolist()
accuracy = df_comparison['Accuracy (%)'].tolist()
f1_score = df_comparison['F1-Score (%)'].tolist()

# Color scheme
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

# Accuracy bar chart
fig.add_trace(
    go.Bar(
        x=stages,
        y=accuracy,
        name='Accuracy',
        marker_color=colors[:len(stages)],
        text=[f'{acc:.2f}%' for acc in accuracy],
        textposition='outside',
        showlegend=False
    ),
    row=1, col=1
)

# F1-Score bar chart
fig.add_trace(
    go.Bar(
        x=stages,
        y=f1_score,
        name='F1-Score',
        marker_color=colors[:len(stages)],
        text=[f'{f1:.2f}%' for f1 in f1_score],
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

fig.update_xaxes(title_text="Stage", row=1, col=1)
fig.update_xaxes(title_text="Stage", row=1, col=2)
fig.update_yaxes(title_text="Accuracy (%)", range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text="F1-Score (%)", range=[0, 100], row=1, col=2)

fig.update_layout(
    title_text="🎯 Stage Performance Comparison",
    height=500,
    font=dict(size=12)
)

fig.show()

## ⏱️ Visualization 2: Training Time vs Accuracy Trade-off

In [ ]:
# Scatter plot: Training time vs Accuracy
fig = go.Figure()

for i, row in df_comparison.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['Training Time (min)']],
        y=[row['Accuracy (%)']],
        mode='markers+text',
        name=row['Stage'],
        marker=dict(size=20, color=colors[i]),
        text=row['Stage'],
        textposition='top center',
        textfont=dict(size=10),
        showlegend=True
    ))

fig.update_layout(
    title='⏱️ Training Time vs Accuracy Trade-off',
    xaxis_title='Training Time (minutes)',
    yaxis_title='Accuracy (%)',
    yaxis=dict(range=[60, 95]),
    height=600,
    hovermode='closest'
)

# Add diagonal reference line
fig.add_annotation(
    text="Ideal: High Accuracy, Low Time",
    xref="paper", yref="paper",
    x=0.05, y=0.95,
    showarrow=False,
    font=dict(size=12, color="gray")
)

fig.show()

## 📊 Visualization 3: Dataset Size Impact

In [ ]:
# Bar chart: Dataset size vs Accuracy
fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_comparison['Stage'],
    y=df_comparison['Train Size'],
    name='Train Size',
    marker_color='lightblue',
    yaxis='y',
    offsetgroup=1
))

fig.add_trace(go.Scatter(
    x=df_comparison['Stage'],
    y=df_comparison['Accuracy (%)'],
    name='Accuracy',
    marker=dict(size=15, color='red'),
    mode='lines+markers',
    yaxis='y2'
))

fig.update_layout(
    title='📊 Dataset Size vs Accuracy',
    xaxis_title='Stage',
    yaxis=dict(title='Training Samples', side='left'),
    yaxis2=dict(
        title='Accuracy (%)',
        overlaying='y',
        side='right',
        range=[60, 95]
    ),
    height=600,
    hovermode='x unified'
)

fig.show()

## 🎯 Visualization 4: Comprehensive Radar Chart

In [ ]:
# Normalize metrics for radar chart (0-100 scale)
radar_data = []

for i, row in df_comparison.iterrows():
    # Normalize training time (inverse: faster = better)
    max_time = df_comparison['Training Time (min)'].max()
    time_score = 100 * (1 - row['Training Time (min)'] / max_time)
    
    # Dataset size score
    max_size = df_comparison['Train Size'].max()
    size_score = 100 * row['Train Size'] / max_size
    
    radar_data.append({
        'Stage': row['Stage'],
        'Accuracy': row['Accuracy (%)'],
        'F1-Score': row['F1-Score (%)'],
        'Speed': time_score,  # Inverse of time
        'Data Size': size_score
    })

# Create radar chart
categories = ['Accuracy', 'F1-Score', 'Speed\n(Inverse Time)', 'Data Size']

fig = go.Figure()

for i, data in enumerate(radar_data):
    fig.add_trace(go.Scatterpolar(
        r=[data['Accuracy'], data['F1-Score'], data['Speed'], data['Data Size']],
        theta=categories,
        fill='toself',
        name=data['Stage'],
        marker=dict(color=colors[i])
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100]
        )
    ),
    title='🎯 Comprehensive Performance Radar Chart',
    height=600,
    showlegend=True
)

fig.show()

## 📋 Detailed Statistical Analysis

In [ ]:
# Calculate improvements and differences
print("=" * 100)
print("📊 DETAILED STATISTICAL ANALYSIS")
print("=" * 100)

# Find best and worst performers
best_acc = df_comparison.loc[df_comparison['Accuracy (%)'].idxmax()]
best_f1 = df_comparison.loc[df_comparison['F1-Score (%)'].idxmax()]
fastest = df_comparison.loc[df_comparison['Training Time (min)'].idxmin()]

print(f"\n🏆 BEST PERFORMERS:")
print(f"   Highest Accuracy: {best_acc['Stage']} - {best_acc['Accuracy (%)']:.2f}%")
print(f"   Highest F1-Score: {best_f1['Stage']} - {best_f1['F1-Score (%)']:.2f}%")
print(f"   Fastest Training: {fastest['Stage']} - {fastest['Training Time (min)']:.1f} minutes")

# Calculate improvements from Stage 1 to others
if 'stage1' in results:
    stage1_acc = df_comparison[df_comparison['Stage'].str.contains('Weak')]['Accuracy (%)'].values[0]
    stage1_f1 = df_comparison[df_comparison['Stage'].str.contains('Weak')]['F1-Score (%)'].values[0]
    
    print(f"\n📈 IMPROVEMENTS OVER STAGE 1 (WEAK SUPERVISION):")
    print(f"   Stage 1 Baseline - Accuracy: {stage1_acc:.2f}%, F1: {stage1_f1:.2f}%")
    
    for _, row in df_comparison.iterrows():
        if not 'Weak' in row['Stage']:
            acc_improve = row['Accuracy (%)'] - stage1_acc
            f1_improve = row['F1-Score (%)'] - stage1_f1
            time_ratio = row['Training Time (min)'] / fastest['Training Time (min)']
            
            print(f"\n   {row['Stage']}:")
            print(f"      Accuracy: +{acc_improve:.2f}% ({acc_improve/stage1_acc*100:.1f}% relative)")
            print(f"      F1-Score: +{f1_improve:.2f}% ({f1_improve/stage1_f1*100:.1f}% relative)")
            print(f"      Time: {time_ratio:.1f}x slower than fastest")

# Cost-benefit analysis
print(f"\n💰 COST-BENEFIT ANALYSIS:")
print("   (Accuracy gain per minute of training time)")
for _, row in df_comparison.iterrows():
    if not 'Weak' in row['Stage']:
        acc_gain = row['Accuracy (%)'] - stage1_acc
        time_cost = row['Training Time (min)'] - fastest['Training Time (min)']
        if time_cost > 0:
            efficiency = acc_gain / time_cost
            print(f"      {row['Stage']}: {efficiency:.2f}% accuracy per extra minute")

print("=" * 100)

## 💡 Recommendations & Insights

In [ ]:
print("=" * 100)
print("💡 KEY INSIGHTS & RECOMMENDATIONS")
print("=" * 100)

# Determine best approach for different scenarios
print("\n🎯 RECOMMENDED STAGE FOR DIFFERENT SCENARIOS:\n")

print("1️⃣ WHEN TO USE STAGE 1 (WEAK SUPERVISION):")
print("   ✅ No labeled data available")
print("   ✅ Fast prototyping needed (< 10 minutes)")
print("   ✅ Gaming community focus (Reddit signals)")
print("   ✅ Limited computational resources")
print("   ✅ Cold start problem / Bootstrap")
print(f"   📊 Performance: {stage1_acc:.2f}% accuracy")

print("\n2️⃣ WHEN TO USE STAGE 2 (BALANCED SUPERVISED):")
stage2_row = df_comparison[df_comparison['Stage'].str.contains('Balanced')].iloc[0]
print("   ✅ Need balanced class performance")
print("   ✅ Fast training with good accuracy")
print("   ✅ Equal false positive/negative cost")
print("   ✅ Baseline supervised approach")
print(f"   📊 Performance: {stage2_row['Accuracy (%)']:.2f}% accuracy in {stage2_row['Training Time (min)']:.1f} min")

print("\n3️⃣ WHEN TO USE STAGE 3A (FOCAL LOSS):")
if 'stage3_focal' in results:
    stage3a_row = df_comparison[df_comparison['Stage'].str.contains('Focal')].iloc[0]
    print("   ✅ High accuracy critical (production)")
    print("   ✅ Imbalanced dataset (natural distribution)")
    print("   ✅ Focus on hard-to-classify examples")
    print("   ✅ Sufficient training time & GPU")
    print(f"   📊 Performance: {stage3a_row['Accuracy (%)']:.2f}% accuracy in {stage3a_row['Training Time (min)']:.1f} min")

print("\n4️⃣ WHEN TO USE STAGE 3B (CLASS WEIGHTING):")
if 'stage3_weighted' in results:
    stage3b_row = df_comparison[df_comparison['Stage'].str.contains('Weighting')].iloc[0]
    print("   ✅ Alternative to Focal Loss")
    print("   ✅ Simpler implementation")
    print("   ✅ Imbalanced classes")
    print("   ✅ Compare with Focal Loss performance")
    print(f"   📊 Performance: {stage3b_row['Accuracy (%)']:.2f}% accuracy in {stage3b_row['Training Time (min)']:.1f} min")

print("\n🔄 HYBRID APPROACH RECOMMENDATION:")
print("   Stage 1 (Bootstrap) → Human Verify → Stage 3 (Production)")
print("   Benefits:")
print("      ⚡ Fast initial deployment (8-10 min)")
print("      🎯 High production accuracy (86%+)")
print("      💰 Reduced manual labeling cost")
print("      🎮 Gaming expertise maintained")

print("\n" + "=" * 100)

## 📊 Stage 3 Comparison: Focal Loss vs Class Weighting

In [ ]:
# Compare Stage 3 variants if both available
if 'stage3_focal' in results and 'stage3_weighted' in results:
    print("=" * 80)
    print("🔍 STAGE 3 DETAILED COMPARISON: FOCAL LOSS vs CLASS WEIGHTING")
    print("=" * 80)
    
    focal_data = results['stage3_focal']
    weighted_data = results['stage3_weighted']
    
    comparison_3 = pd.DataFrame({
        'Metric': [
            'Accuracy (%)',
            'F1-Score (%)',
            'Training Time (min)',
            'Dataset Size',
            'Loss Function',
            'Approach'
        ],
        'Focal Loss': [
            focal_data['metrics']['accuracy'] * 100,
            focal_data['metrics']['f1_weighted'] * 100,
            focal_data.get('training_time_seconds', 0) / 60,
            focal_data.get('dataset_size', 0),
            f"α={focal_data.get('focal_loss_params', {}).get('alpha', 'N/A')}, γ={focal_data.get('focal_loss_params', {}).get('gamma', 'N/A')}",
            'Down-weight easy examples'
        ],
        'Class Weighting': [
            weighted_data['metrics']['accuracy'] * 100,
            weighted_data['metrics']['f1_weighted'] * 100,
            weighted_data.get('training_time_seconds', 0) / 60,
            weighted_data.get('dataset_size', 0),
            weighted_data.get('loss_function', 'CrossEntropy + Weights'),
            'Weight minority classes'
        ]
    })
    
    display(comparison_3)
    
    # Calculate difference
    acc_diff = focal_data['metrics']['accuracy'] - weighted_data['metrics']['accuracy']
    f1_diff = focal_data['metrics']['f1_weighted'] - weighted_data['metrics']['f1_weighted']
    
    print(f"\n📊 Performance Difference:")
    print(f"   Accuracy: {abs(acc_diff)*100:.2f}% {'(Focal Loss better)' if acc_diff > 0 else '(Class Weighting better)'}")
    print(f"   F1-Score: {abs(f1_diff)*100:.2f}% {'(Focal Loss better)' if f1_diff > 0 else '(Class Weighting better)'}")
    
    print("\n💡 Insights:")
    if abs(acc_diff) < 0.01:  # Less than 1% difference
        print("   ⚖️ Both methods perform similarly on this dataset")
        print("   🎯 Choose based on implementation preference")
    elif acc_diff > 0:
        print("   🏆 Focal Loss shows better performance")
        print("   📈 Better at handling hard examples")
    else:
        print("   🏆 Class Weighting shows better performance")
        print("   ⚡ Simpler approach with good results")
    
    print("=" * 80)
else:
    print("⚠️ Both Stage 3 variants not available yet for comparison")

## 💾 Export Comparison Results

In [ ]:
# Save comparison table
df_comparison.to_csv('stage4_comparison_table.csv', index=False)
print("💾 Saved: stage4_comparison_table.csv")

# Save detailed analysis
analysis_summary = {
    'comparison_summary': df_comparison.to_dict('records'),
    'best_performers': {
        'highest_accuracy': {
            'stage': best_acc['Stage'],
            'value': float(best_acc['Accuracy (%)']),
        },
        'highest_f1': {
            'stage': best_f1['Stage'],
            'value': float(best_f1['F1-Score (%)']),
        },
        'fastest_training': {
            'stage': fastest['Stage'],
            'value': float(fastest['Training Time (min)']),
        }
    },
    'recommendations': {
        'rapid_prototyping': 'Stage 1 (Weak Supervision)',
        'balanced_performance': 'Stage 2 (Balanced Supervised)',
        'highest_accuracy': 'Stage 3 (Focal Loss or Class Weighting)',
        'hybrid_approach': 'Stage 1 → Human Verify → Stage 3'
    }
}

with open('stage4_analysis_summary.json', 'w') as f:
    json.dump(analysis_summary, f, indent=2)

print("💾 Saved: stage4_analysis_summary.json")

# Download files
print("\n📥 Downloading result files...")
files.download('stage4_comparison_table.csv')
files.download('stage4_analysis_summary.json')

print("\n✅ Stage 4 Analysis Complete!")
print("=" * 80)
print("🎉 ALL STAGES COMPARED SUCCESSFULLY!")
print("=" * 80)

## 📝 Conclusion

### 🏆 Key Findings:

1. **Accuracy Ranking**: Stage 3 > Stage 2 > Stage 1
2. **Speed Ranking**: Stage 1 > Stage 2 > Stage 3
3. **Cost Ranking**: Stage 1 (no labels) < Stage 2 & 3 (labeled data)

### 💡 Best Practices:

- **Prototyping**: Start with Stage 1 (8-10 min, 69% accuracy)
- **Baseline**: Use Stage 2 (10 min, 85% accuracy)
- **Production**: Deploy Stage 3 (109 min, 87% accuracy)
- **Optimal**: Hybrid approach combining all stages

### 🚀 Future Work:

- [ ] Ensemble methods (combine all models)
- [ ] Active learning pipeline
- [ ] Real-time inference optimization
- [ ] Multi-language support
- [ ] API deployment

---

**Thank you for completing all 4 stages! 🎉**